# Indirect prompt injection: training + evaluation on Colab

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. Total ≈ 10–15 min.

At the end two files download: `colab_results.zip` (small, send it back) and optionally the DistilBERT weights.

In [ ]:
from google.colab import files
up = files.upload()  # choose prompt-injection-detector.zip
!unzip -q -o prompt-injection-detector.zip && ls

In [ ]:
%cd /content/prompt-injection-detector
!pip install -q scikit-learn==1.9.0 sentence-transformers nltk accelerate
import torch; print('GPU:', torch.cuda.is_available())

## 1. Re-create the direct-trained DistilBERT (weights were never committed)

In [ ]:
!python -m src.models.distilbert_classifier

## 2. Train all three models on direct + indirect data

In [ ]:
!python -m scripts.train_indirect

## 3. Full evaluation (4 test sets, doc/chunk/decode, adversarial, localisation)

In [ ]:
!python -m scripts.eval_indirect
!head -60 results/indirect/indirect_results.md

## 4. Explainability (TF-IDF terms, window profiles, DistilBERT attention)

In [ ]:
!python -m scripts.explain_indirect

## 5. Download results (send `colab_results.zip` back)

In [ ]:
!zip -q -r colab_results.zip results/indirect models/tfidf_lr_indirect.joblib models/embed_lr_indirect.joblib
files.download('colab_results.zip')

## 6. Optional: DistilBERT weights for the laptop demo (~500 MB)

In [ ]:
!zip -q -r distilbert_weights.zip models/distilbert models/distilbert_indirect -x '*checkpoints*'
files.download('distilbert_weights.zip')